# Évaluation du module RAG : retrieval documentaire

Ce notebook évalue la capacité du module RAG à retrouver les bons passages documentaires à partir de questions utilisateur.

Objectifs :

- tester le retriever ChromaDB avec les vrais embeddings ;
- observer les sources retrouvées pour chaque question ;
- mesurer les scores de similarité ;
- vérifier que les documents récupérés contiennent les thèmes attendus ;
- produire des résultats exploitables dans le rapport de stage.

Contrairement au notebook `01_intent_routing_evaluation`, ce notebook utilise le vrai module RAG du projet.

## 1. Initialisation

On ajoute la racine du projet au `PYTHONPATH`. Le notebook doit être exécuté avec le kernel du venv du projet.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 2. Vérification des fichiers RAG

Le dossier `data/vectorstore/` n'est pas versionné. Si l'index local n'existe pas, il faut relancer :

```bash
python scripts/ingest_rag_sources.py
python scripts/build_rag_index.py
```

In [ ]:
from backend.app.core.config import RAG_CHUNKS_PATH, RAG_VECTORSTORE_PATH, RAG_COLLECTION_NAME

print("Chunks path:", RAG_CHUNKS_PATH)
print("Vectorstore path:", RAG_VECTORSTORE_PATH)
print("Collection:", RAG_COLLECTION_NAME)

assert RAG_CHUNKS_PATH.exists(), "chunks.jsonl introuvable. Lance scripts/ingest_rag_sources.py"
assert RAG_VECTORSTORE_PATH.exists(), "Vectorstore Chroma introuvable. Lance scripts/build_rag_index.py"

print("Fichiers RAG trouvés.")

## 3. Imports du retriever

In [ ]:
import json
import time
import pandas as pd

from backend.app.rag.retriever import search

print("Retriever importé.")

## 4. Jeu de questions d'évaluation

Chaque question est associée à des mots-clés attendus. L'objectif n'est pas de prouver une vérité métier complète, mais de vérifier que le retrieval retourne des passages cohérents avec le thème demandé.

In [ ]:
test_cases = [
    {
        "question": "Comment faire opposition à une carte ?",
        "expected_keywords": ["opposition", "carte", "confirmation"],
        "expected_source_hint": "Opposition sur carte",
        "category": "carte",
    },
    {
        "question": "Quelles actions nécessitent une confirmation ?",
        "expected_keywords": ["confirmation", "actions sensibles", "virement", "opposition"],
        "expected_source_hint": "Confirmation des opérations sensibles",
        "category": "securite",
    },
    {
        "question": "Comment commander un chéquier ?",
        "expected_keywords": ["chéquier", "chequier", "demande", "confirmation"],
        "expected_source_hint": "chéquier",
        "category": "service",
    },
    {
        "question": "Comment demander un relevé de compte ?",
        "expected_keywords": ["relevé", "releve", "document", "compte"],
        "expected_source_hint": "document",
        "category": "service",
    },
    {
        "question": "Comment faire un virement ?",
        "expected_keywords": ["virement", "compte", "bénéficiaire", "beneficiaire", "confirmation"],
        "expected_source_hint": "virement",
        "category": "virement",
    },
    {
        "question": "Quels services sont disponibles sur AMENet ?",
        "expected_keywords": ["amenet", "solde", "mouvements", "virement", "carte"],
        "expected_source_hint": "services",
        "category": "amenet",
    },
    {
        "question": "Comment consulter mes mouvements bancaires ?",
        "expected_keywords": ["mouvements", "transactions", "compte", "historique"],
        "expected_source_hint": "mouvements",
        "category": "consultation",
    },
    {
        "question": "Comment contacter le support AMENet ?",
        "expected_keywords": ["support", "messagerie", "assistance", "amenet"],
        "expected_source_hint": "support",
        "category": "support",
    },
]

len(test_cases)

## 5. Fonctions d'évaluation

In [ ]:
import unicodedata


def normalize(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(char for char in text if unicodedata.category(char) != "Mn")
    return text


def contains_any(text: str, keywords: list[str]) -> bool:
    normalized_text = normalize(text)
    return any(normalize(keyword) in normalized_text for keyword in keywords)


def count_keyword_hits(text: str, keywords: list[str]) -> int:
    normalized_text = normalize(text)
    return sum(1 for keyword in keywords if normalize(keyword) in normalized_text)


def format_top_titles(results) -> str:
    return " | ".join(result.title for result in results)


def format_top_sources(results) -> str:
    sources = []
    for result in results:
        source = result.source_file
        if result.page is not None:
            source += f":page {result.page}"
        sources.append(source)
    return " | ".join(sources)


print("Fonctions d'évaluation prêtes.")

## 6. Exécution de l'évaluation retrieval

In [ ]:
rows = []
top_k = 4

for index, case in enumerate(test_cases, start=1):
    start_time = time.perf_counter()
    results = search(case["question"], top_k=top_k)
    elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)

    top1 = results[0] if results else None
    joined_top_text = "\n".join(
        f"{result.title}\n{result.text}" for result in results
    )

    keyword_hits = count_keyword_hits(joined_top_text, case["expected_keywords"])
    top3_has_expected_keywords = keyword_hits > 0
    top1_has_expected_keywords = False

    if top1:
        top1_joined = f"{top1.title}\n{top1.text}"
        top1_has_expected_keywords = contains_any(top1_joined, case["expected_keywords"])

    retrieval_ok = top3_has_expected_keywords

    rows.append(
        {
            "id": index,
            "category": case["category"],
            "question": case["question"],
            "expected_keywords": ", ".join(case["expected_keywords"]),
            "expected_source_hint": case["expected_source_hint"],
            "top1_title": top1.title if top1 else None,
            "top1_score": round(top1.score, 3) if top1 and top1.score is not None else None,
            "top1_distance": round(top1.distance, 3) if top1 and top1.distance is not None else None,
            "top1_source_file": top1.source_file if top1 else None,
            "top1_page": top1.page if top1 else None,
            "top1_source_image": top1.source_image if top1 else None,
            "top_k": len(results),
            "top_titles": format_top_titles(results),
            "top_sources": format_top_sources(results),
            "keyword_hits_topk": keyword_hits,
            "top1_has_expected_keywords": top1_has_expected_keywords,
            "topk_has_expected_keywords": top3_has_expected_keywords,
            "retrieval_ok": retrieval_ok,
            "latency_ms": elapsed_ms,
            "top1_preview": top1.text[:300] if top1 else None,
        }
    )

df = pd.DataFrame(rows)
df

## 7. Résumé global

In [ ]:
total = len(df)
successes = int(df["retrieval_ok"].sum())
accuracy = successes / total if total else 0

summary = pd.DataFrame(
    [
        {"metric": "Nombre de questions", "value": total},
        {"metric": "Questions avec retrieval cohérent", "value": successes},
        {"metric": "Taux de réussite top-k", "value": round(accuracy, 3)},
        {"metric": "Latence moyenne retrieval ms", "value": round(df["latency_ms"].mean(), 2)},
        {"metric": "Score top-1 moyen", "value": round(df["top1_score"].mean(), 3)},
    ]
)

summary

## 8. Résultats par catégorie

In [ ]:
category_summary = (
    df.groupby("category")
    .agg(
        questions=("id", "count"),
        successes=("retrieval_ok", "sum"),
        mean_top1_score=("top1_score", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)

category_summary["success_rate"] = (
    category_summary["successes"] / category_summary["questions"]
).round(3)
category_summary["mean_top1_score"] = category_summary["mean_top1_score"].round(3)
category_summary["mean_latency_ms"] = category_summary["mean_latency_ms"].round(2)

category_summary

## 9. Analyse des échecs éventuels

In [ ]:
failures = df[~df["retrieval_ok"]]

if failures.empty:
    print("Tous les cas ont retrouvé au moins un passage contenant les thèmes attendus dans le top-k.")
else:
    display(
        failures[
            [
                "question",
                "expected_keywords",
                "top1_title",
                "top1_score",
                "top_titles",
                "top1_preview",
            ]
        ]
    )

## 10. Détail des sources retrouvées

Ce tableau est utile pour le rapport, car il montre quelles sections documentaires ont été récupérées par le RAG.

In [ ]:
df[
    [
        "question",
        "top1_title",
        "top1_score",
        "top1_source_file",
        "top1_page",
        "top1_source_image",
        "retrieval_ok",
    ]
]

## 11. Export des résultats

Les résultats sont exportés au format CSV, JSON et Markdown pour pouvoir être réutilisés dans le rapport de stage.

In [ ]:
evaluation_dir = PROJECT_ROOT / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

csv_path = evaluation_dir / "rag_retrieval_evaluation.csv"
json_path = evaluation_dir / "rag_retrieval_evaluation.json"
md_path = evaluation_dir / "rag_retrieval_evaluation.md"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2, force_ascii=False)

report_columns = [
    "question",
    "top1_title",
    "top1_score",
    "keyword_hits_topk",
    "retrieval_ok",
]

markdown_report = "# Résultats de l'évaluation RAG retrieval\n\n"
markdown_report += summary.to_markdown(index=False)
markdown_report += "\n\n## Détail par question\n\n"
markdown_report += df[report_columns].to_markdown(index=False)

md_path.write_text(markdown_report, encoding="utf-8")

print(f"Résultats CSV : {csv_path}")
print(f"Résultats JSON : {json_path}")
print(f"Rapport Markdown : {md_path}")

## 12. Conclusion

Cette évaluation vérifie la capacité du RAG à retrouver des passages pertinents avant la génération de réponse.

Les métriques utilisées restent simples : score top-1, présence de mots-clés attendus dans le top-k et latence de retrieval. Elles permettent toutefois de documenter la robustesse du prototype et d'identifier les questions pour lesquelles la base documentaire doit être enrichie.